# Building Efficient Data Pipelines

**What you'll learn in this notebook:**
- How PyTorch's `Dataset` and `DataLoader` work together
- Creating custom datasets for any data source
- Tuning DataLoader parameters for performance
- Handling variable-length sequences with custom collate functions
- Data augmentation and MixUp from scratch
- Proper train/val/test splitting patterns

**Prerequisites:** Basic PyTorch tensor operations, familiarity with Python classes.

**All code runs on CPU — no GPU required.**

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import torch.nn.functional as F
print(f"PyTorch version: {torch.__version__}")

## The Data Loading Pipeline

PyTorch's data loading follows a simple two-part design:

1. **Dataset** — Defines *what* your data is (how to access individual samples)
2. **DataLoader** — Defines *how* to serve data (batching, shuffling, parallelism)

```
Dataset (random access to samples)
    ↓
DataLoader (batching + shuffling + parallelism)
    ↓
Training loop (gets mini-batches)
```

Let's start with the simplest possible dataset.

## TensorDataset — The Quickest Start

When your data already fits in memory as tensors, `TensorDataset` wraps them directly. It pairs up corresponding elements from multiple tensors (like `zip`).

In [ ]:
# Create synthetic regression data: y = 2x + 1 + noise
X = torch.randn(1000, 3)  # 1000 samples, 3 features
y = X @ torch.tensor([2.0, -1.0, 0.5]) + 1.0 + torch.randn(1000) * 0.1

# Wrap in TensorDataset
dataset = TensorDataset(X, y)

print(f"Dataset length: {len(dataset)}")
print(f"Single sample: features shape={dataset[0][0].shape}, target shape={dataset[0][1].shape}")
print(f"First sample: X={dataset[0][0]}, y={dataset[0][1]:.4f}")

In [ ]:
# Create a DataLoader to iterate in batches
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Iterate one batch
batch_X, batch_y = next(iter(loader))
print(f"Batch shapes: X={batch_X.shape}, y={batch_y.shape}")
print(f"Number of batches per epoch: {len(loader)}")

# Typical training loop pattern
for epoch in range(2):
    total_loss = 0.0
    for batch_X, batch_y in loader:
        # In real training: forward pass, loss, backward, step
        loss = F.mse_loss(batch_X.sum(dim=1), batch_y)
        total_loss += loss.item()
    print(f"Epoch {epoch}: avg batch loss = {total_loss / len(loader):.4f}")

## Custom Dataset

For most real applications, you need a custom `Dataset`. The contract is simple — implement two methods:
- `__len__()` — return the total number of samples
- `__getitem__(idx)` — return the sample at index `idx`

This is all PyTorch needs to index into your data.

In [ ]:
class SyntheticClassificationDataset(Dataset):
    """A custom dataset generating random classification data on-the-fly."""
    
    def __init__(self, num_samples=500, num_features=10, num_classes=3):
        self.num_samples = num_samples
        self.num_features = num_features
        # Pre-generate all data (common for in-memory datasets)
        self.features = torch.randn(num_samples, num_features)
        self.labels = torch.randint(0, num_classes, (num_samples,))
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


dataset = SyntheticClassificationDataset(num_samples=200, num_features=5, num_classes=4)
print(f"Dataset size: {len(dataset)}")
print(f"Sample 0: features={dataset[0][0].shape}, label={dataset[0][1]}")

# Slicing works via DataLoader, not directly on Dataset
loader = DataLoader(dataset, batch_size=16)
features, labels = next(iter(loader))
print(f"\nBatch: features={features.shape}, labels={labels.shape}")
print(f"Label distribution in batch: {torch.bincount(labels, minlength=4)}")

## DataLoader Parameters Explained

The `DataLoader` has many knobs for controlling performance and behavior. Let's explore the most important ones.

In [ ]:
dataset = TensorDataset(torch.arange(100).float(), torch.arange(100).float())

# batch_size: number of samples per batch
loader = DataLoader(dataset, batch_size=30)
batch_sizes = [len(batch[0]) for batch in loader]
print(f"batch_size=30, dataset=100: batch sizes = {batch_sizes}")
# Notice: last batch has only 10 samples!

# drop_last=True: discard incomplete final batch (useful for BatchNorm)
loader = DataLoader(dataset, batch_size=30, drop_last=True)
batch_sizes = [len(batch[0]) for batch in loader]
print(f"batch_size=30, drop_last=True: batch sizes = {batch_sizes}")

# shuffle=True: randomize order each epoch (critical for training!)
loader = DataLoader(dataset, batch_size=5, shuffle=True)
first_batch = next(iter(loader))
print(f"\nshuffle=True, first batch indices: {first_batch[0][:5].long().tolist()}")

loader2 = DataLoader(dataset, batch_size=5, shuffle=True)
first_batch2 = next(iter(loader2))
print(f"shuffle=True, next epoch first batch: {first_batch2[0][:5].long().tolist()}")
print("(Different order each time!)")

### num_workers and pin_memory

- **`num_workers`**: Number of subprocess workers for loading data in parallel. Set to 0 for in-process loading (default). A good starting value is the number of CPU cores.
- **`pin_memory`**: When True, DataLoader copies tensors into pinned (page-locked) memory before returning them. This speeds up CPU→GPU transfers. Only useful when training on GPU.

```python
# Typical production settings (on GPU machine):
loader = DataLoader(dataset, batch_size=64, shuffle=True,
                    num_workers=4, pin_memory=True,
                    persistent_workers=True)  # keep workers alive between epochs
```

In [ ]:
import time

big_dataset = TensorDataset(torch.randn(10000, 100), torch.randint(0, 10, (10000,)))

# Compare iteration speed (on CPU, num_workers > 0 has overhead for simple data)
for num_workers in [0, 2]:
    loader = DataLoader(big_dataset, batch_size=128, num_workers=num_workers)
    start = time.perf_counter()
    for batch in loader:
        pass
    elapsed = time.perf_counter() - start
    print(f"num_workers={num_workers}: {elapsed:.3f}s to iterate full dataset")

print("\nNote: For in-memory tensor data, num_workers=0 is often fastest.")
print("num_workers > 0 shines when __getitem__ does I/O (reading files, decoding images).")

## Variable-Length Data with Custom collate_fn

Real data isn't always uniform. Text sequences, time series, and graphs have variable lengths. The default `collate_fn` tries to stack tensors — but this fails when dimensions don't match.

A **custom collate function** tells the DataLoader how to combine variable-length samples into a batch.

In [ ]:
class VariableLengthSequenceDataset(Dataset):
    """Simulates a text dataset where sequences have different lengths."""
    
    def __init__(self, num_samples=100, max_len=20, vocab_size=50):
        self.sequences = []
        self.labels = []
        for _ in range(num_samples):
            length = torch.randint(3, max_len + 1, (1,)).item()
            seq = torch.randint(0, vocab_size, (length,))
            label = torch.tensor(1 if length > max_len // 2 else 0)
            self.sequences.append(seq)
            self.labels.append(label)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


seq_dataset = VariableLengthSequenceDataset(num_samples=50)
print("Sample lengths:", [len(seq_dataset[i][0]) for i in range(5)])

# This will FAIL with default collate:
try:
    loader = DataLoader(seq_dataset, batch_size=4)
    batch = next(iter(loader))
except RuntimeError as e:
    print(f"\nDefault collate fails: {e}")

In [ ]:
def pad_collate_fn(batch):
    """Pad sequences to the length of the longest in the batch."""
    sequences, labels = zip(*batch)
    
    # Find max length in this batch
    lengths = torch.tensor([len(seq) for seq in sequences])
    max_len = lengths.max().item()
    
    # Pad all sequences to max_len (pad with 0)
    padded = torch.zeros(len(sequences), max_len, dtype=sequences[0].dtype)
    for i, seq in enumerate(sequences):
        padded[i, :len(seq)] = seq
    
    labels = torch.stack(labels)
    return padded, labels, lengths  # Return lengths for masking later


loader = DataLoader(seq_dataset, batch_size=4, collate_fn=pad_collate_fn, shuffle=True)
padded_seqs, labels, lengths = next(iter(loader))

print(f"Padded batch shape: {padded_seqs.shape}")
print(f"Lengths: {lengths.tolist()}")
print(f"Labels: {labels.tolist()}")
print(f"\nFirst sequence (padded): {padded_seqs[0].tolist()}")
print(f"Actual length: {lengths[0].item()} (rest is padding zeros)")

## Data Augmentation Patterns

Data augmentation creates diversity in your training data by applying random transformations. For tensors (as opposed to images), common augmentations include noise injection, random masking, and feature dropout.

In [ ]:
class AugmentedDataset(Dataset):
    """Wraps a dataset and applies random augmentations during training."""
    
    def __init__(self, base_dataset, augment=True, noise_std=0.1, mask_prob=0.15):
        self.base_dataset = base_dataset
        self.augment = augment
        self.noise_std = noise_std
        self.mask_prob = mask_prob
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        features, label = self.base_dataset[idx]
        
        if self.augment:
            # Gaussian noise injection
            features = features + torch.randn_like(features) * self.noise_std
            
            # Random feature masking (set random features to 0)
            mask = torch.rand_like(features) > self.mask_prob
            features = features * mask
        
        return features, label


base = SyntheticClassificationDataset(num_samples=100, num_features=8)
train_ds = AugmentedDataset(base, augment=True)
val_ds = AugmentedDataset(base, augment=False)  # No augmentation for validation!

# Same underlying sample, different augmented views
print("Original:", base[0][0][:4])
print("Augmented view 1:", train_ds[0][0][:4])
print("Augmented view 2:", train_ds[0][0][:4])
print("Validation (no aug):", val_ds[0][0][:4])

## MixUp From Scratch

MixUp is a powerful regularization technique that creates virtual training samples by linearly interpolating between pairs of examples and their labels.

Given two samples $(x_i, y_i)$ and $(x_j, y_j)$:
$$\tilde{x} = \lambda x_i + (1 - \lambda) x_j$$
$$\tilde{y} = \lambda y_i + (1 - \lambda) y_j$$

where $\lambda \sim \text{Beta}(\alpha, \alpha)$.

In [ ]:
def mixup_batch(x, y, alpha=0.2):
    """Apply MixUp to a batch of data.
    
    Args:
        x: input features [batch_size, ...]
        y: one-hot labels [batch_size, num_classes]
        alpha: Beta distribution parameter (higher = more mixing)
    
    Returns:
        mixed_x, mixed_y
    """
    batch_size = x.size(0)
    
    # Sample mixing coefficient from Beta distribution
    lam = torch.distributions.Beta(alpha, alpha).sample((batch_size, 1))
    
    # Random permutation for pairing
    perm = torch.randperm(batch_size)
    
    # Mix features (reshape lam for broadcasting)
    lam_x = lam.expand_as(x)
    mixed_x = lam_x * x + (1 - lam_x) * x[perm]
    
    # Mix labels
    lam_y = lam.expand_as(y)
    mixed_y = lam_y * y + (1 - lam_y) * y[perm]
    
    return mixed_x, mixed_y


# Demo MixUp
num_classes = 4
x = torch.randn(8, 5)  # 8 samples, 5 features
y_indices = torch.randint(0, num_classes, (8,))
y_onehot = F.one_hot(y_indices, num_classes).float()

mixed_x, mixed_y = mixup_batch(x, y_onehot, alpha=0.4)

print("Original labels (one-hot):")
print(y_onehot[:3])
print("\nMixed labels (soft targets):")
print(mixed_y[:3])
print("\nNotice: mixed labels are no longer one-hot — they're soft targets!")

## Train/Val/Test Split Patterns

Never evaluate on training data! PyTorch provides `random_split` for easy splitting, but you can also do it manually for more control.

In [ ]:
# Method 1: random_split (simplest)
full_dataset = TensorDataset(torch.randn(1000, 10), torch.randint(0, 2, (1000,)))

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)  # Reproducible splits!
)

print(f"Total: {len(full_dataset)} → Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# Create loaders with different settings for each split
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)  # No shuffle for val!
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(f"\nTrain batches: {len(train_loader)} (shuffled, drop_last)")
print(f"Val batches: {len(val_loader)} (ordered, keep all)")
print(f"Test batches: {len(test_loader)} (ordered, keep all)")

In [ ]:
# Method 2: Index-based split (more control, stratified sampling)
from torch.utils.data import Subset

full_dataset = SyntheticClassificationDataset(num_samples=200, num_classes=3)

# Stratified split: maintain class balance in each split
labels = full_dataset.labels
indices_by_class = {}
for idx in range(len(full_dataset)):
    label = labels[idx].item()
    indices_by_class.setdefault(label, []).append(idx)

train_indices, val_indices, test_indices = [], [], []
for class_label, indices in indices_by_class.items():
    perm = torch.randperm(len(indices)).tolist()
    n_train = int(0.7 * len(indices))
    n_val = int(0.15 * len(indices))
    
    train_indices.extend([indices[i] for i in perm[:n_train]])
    val_indices.extend([indices[i] for i in perm[n_train:n_train + n_val]])
    test_indices.extend([indices[i] for i in perm[n_train + n_val:]])

train_ds = Subset(full_dataset, train_indices)
val_ds = Subset(full_dataset, val_indices)
test_ds = Subset(full_dataset, test_indices)

print(f"Stratified split: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

# Verify class balance
train_labels = torch.tensor([full_dataset.labels[i].item() for i in train_indices])
print(f"Train class distribution: {torch.bincount(train_labels).tolist()}")

## Putting It All Together: Complete Training Pipeline

In [ ]:
import torch.nn as nn

# Complete pipeline: Custom Dataset → Augmentation → DataLoader → Training
class SimpleModel(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )
    
    def forward(self, x):
        return self.net(x)

# Setup
dataset = SyntheticClassificationDataset(num_samples=500, num_features=10, num_classes=3)
train_ds, val_ds = random_split(dataset, [400, 100], generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

model = SimpleModel(10, 3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(5):
    model.train()
    train_loss = 0.0
    for features, labels in train_loader:
        logits = model(features)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in val_loader:
            preds = model(features).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)
    
    print(f"Epoch {epoch}: train_loss={train_loss/len(train_loader):.4f}, val_acc={correct/total:.2%}")

## Try It Yourself: Custom CSV-like Dataset

**Exercise:** Create a custom `Dataset` that reads from a dictionary simulating CSV data (columns of mixed types). Your dataset should:
1. Accept a dictionary with keys as column names and values as lists
2. Convert numeric columns to float tensors, categorical columns to integer labels
3. Support a configurable target column

Starter code is below — fill in the `__getitem__` method.

In [ ]:
# Exercise: Complete this dataset class

csv_data = {
    "age": [25, 30, 35, 40, 22, 28, 45, 33, 27, 50],
    "income": [50000, 60000, 75000, 90000, 35000, 55000, 100000, 70000, 48000, 120000],
    "category": ["A", "B", "A", "C", "B", "A", "C", "B", "A", "C"],
    "purchased": [0, 1, 1, 1, 0, 0, 1, 1, 0, 1],  # target
}

class CSVDataset(Dataset):
    def __init__(self, data_dict, target_col, categorical_cols=None):
        self.target = torch.tensor(data_dict[target_col], dtype=torch.long)
        self.categorical_cols = categorical_cols or []
        
        # Build features: encode categoricals, keep numerics
        feature_cols = [k for k in data_dict if k != target_col]
        self.features = []
        
        # Create label encodings for categorical columns
        self.label_maps = {}
        for col in self.categorical_cols:
            unique_vals = sorted(set(data_dict[col]))
            self.label_maps[col] = {v: i for i, v in enumerate(unique_vals)}
        
        # Convert each row to a feature tensor
        for i in range(len(self.target)):
            row = []
            for col in feature_cols:
                if col in self.categorical_cols:
                    row.append(float(self.label_maps[col][data_dict[col][i]]))
                else:
                    row.append(float(data_dict[col][i]))
            self.features.append(torch.tensor(row))
    
    def __len__(self):
        return len(self.target)
    
    def __getitem__(self, idx):
        # TODO: Return (features, target) for the given index
        return self.features[idx], self.target[idx]


# Test your implementation
ds = CSVDataset(csv_data, target_col="purchased", categorical_cols=["category"])
loader = DataLoader(ds, batch_size=4, shuffle=True)
features, targets = next(iter(loader))
print(f"Batch features shape: {features.shape}")
print(f"Batch targets: {targets.tolist()}")
print(f"Feature example (age, income, encoded_category): {features[0].tolist()}")

## Key Takeaways

1. **Dataset** defines access to individual samples; **DataLoader** handles batching and parallelism
2. **TensorDataset** is the fastest start when data fits in memory
3. Custom datasets only need `__len__` and `__getitem__` — keep them simple
4. **shuffle=True** for training, **shuffle=False** for validation/test
5. Use **custom collate_fn** for variable-length data (padding, packing)
6. **num_workers > 0** helps when `__getitem__` is I/O-bound (not for in-memory tensors)
7. Apply augmentation in the dataset's `__getitem__` — disable it for validation
8. **MixUp** creates soft labels by interpolating samples — powerful regularizer
9. Always use a fixed seed for `random_split` to ensure reproducible splits
10. **Stratified splits** maintain class balance across train/val/test